In [36]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

### Trades Data Collection

In [31]:
trades_df = pd.read_parquet("../data/processed/tradesAInvestALL.parquet")
trades_df

,id,size,reporting_gap,name,trade_type,filing_date,state,trade_date,party,ticker
0,H002482383202608110036,$100K-$250K,29 Days,Cory Anthony Booker,sell,2026-09-09,NJ,2026-08-11,Democrat,AMZN
1,H002482383202608110038,$100K-$250K,29 Days,Cory Anthony Booker,sell,2026-09-09,NJ,2026-08-11,Democrat,AMZN
2,H002482383202608110037,$100K-$250K,29 Days,Cory Anthony Booker,sell,2026-09-09,NJ,2026-08-11,Democrat,AMZN
3,H002482383202608110040,$100K-$250K,29 Days,Cory Anthony Booker,sell,2026-09-09,NJ,2026-08-11,Democrat,AMZN
4,H002482383202608110042,$100K-$250K,29 Days,Cory Anthony Booker,sell,2026-09-09,NJ,2026-08-11,Democrat,AMZN
...,...,...,...,...,...,...,...,...,...,...
4417,H002474652202306210013,$50K-$100K,28 Days,Michael T. McCaul,sell,2023-07-19,TX,2023-06-21,Republican,UNH
4418,H002487714202306210013,$1K-$15K,47 Days,John R. Curtis,buy,2023-08-07,UT,2023-06-21,Republican,UNH
4419,H002474652202306160006,$100K-$250K,33 Days,Michael T. McCaul,sell,2023-07-19,TX,2023-06-16,Republican,UNH
4420,H002474652202306160007,$50K-$100K,33 Days,Michael T. McCaul,sell,2023-07-19,TX,2023-06-16,Republican,UNH


In [32]:
# Making ticker name match with yahoo finance data
trades_df["ticker"] = trades_df["ticker"].replace({"BRK.B" : "BRK-B"})

### Stock Data Collection

In [43]:
from pathlib import Path

stock_dir = Path("../data/raw/stock_prices")

stock_files = list(stock_dir.glob("*.parquet"))

print(stock_files)

[PosixPath('../data/raw/stock_prices/AMZN.parquet'), PosixPath('../data/raw/stock_prices/NVDA.parquet'), PosixPath('../data/raw/stock_prices/AAPL.parquet'), PosixPath('../data/raw/stock_prices/META.parquet'), PosixPath('../data/raw/stock_prices/DELL.parquet'), PosixPath('../data/raw/stock_prices/BRK-B.parquet'), PosixPath('../data/raw/stock_prices/SMCI.parquet'), PosixPath('../data/raw/stock_prices/GOOGL.parquet'), PosixPath('../data/raw/stock_prices/MSFT.parquet'), PosixPath('../data/raw/stock_prices/JNJ.parquet'), PosixPath('../data/raw/stock_prices/UNH.parquet')]


In [44]:
stock_data = {}

for file in stock_files:
    ticker = file.stem #.stem takes out .filetype
    stock_data[ticker] = pd.read_parquet(file)

In [46]:
stock_data["AAPL"]

Price,Close,High,Low,Open,Volume
Ticker,AAPL,AAPL,AAPL,AAPL,AAPL
Date,,,,,
2016-01-04,23.688675,23.693173,22.935405,23.072567,270597600
2016-01-05,23.095053,23.801103,23.027597,23.778618,223164000
2016-01-06,22.643091,23.018604,22.456461,22.611611,273829600
2016-01-07,21.687445,22.514918,21.682948,22.188876,324377600
2016-01-08,21.802126,22.285569,21.757155,22.159649,283192000
...,...,...,...,...,...
2026-10-01,330.320007,332.480011,325.809998,330.000000,36306300
2026-10-02,333.690002,334.540009,330.609985,333.260010,33278600


In [47]:
trades_df["trade_date"].head()

0   2026-08-11
1   2026-08-11
2   2026-08-11
3   2026-08-11
4   2026-08-11
Name: trade_date, dtype: datetime64[us]

In [48]:
stock_data["AAPL"].index.dtype

dtype('<M8[ms]')

In [53]:
## Making datetime dtype same
trades_df["trade_date"] = pd.to_datetime(trades_df["trade_date"]).astype("datetime64[us]").dt.normalize() # .dt as series .normalize sets any potential time to 00:00:00 (dont care abt time)
trades_df["filing_date"] = pd.to_datetime(trades_df["filing_date"]).astype("datetime64[us]").dt.normalize() # .dt as series .normalize sets any potential time to 00:00:00 (dont care abt time)

for ticker, df in stock_data.items():
    df.index = pd.to_datetime(df.index).astype("datetime64[us]").normalize()

In [54]:
print(trades_df["trade_date"].dtype)
print(trades_df["filing_date"].dtype)
print(stock_data["AAPL"].index.dtype)

datetime64[us]
datetime64[us]
datetime64[us]


In [ ]:
aapl = stock_data["AAPL"]

matched = trades_df.loc[trades_df["ticker"] == "AAPL", "trade_date"].isin(aapl.index) # using isin just in case filing date is on a non trading day

trade_date
True    578
Name: count, dtype: int64

In [64]:
for ticker, stock in stock_data.items():
    trades = trades_df.loc[
        trades_df["ticker"] == ticker, "trade_date"
    ]

    matched = trades.isin(stock.index)

    print(
        f"{ticker}: {matched.sum()}/{len(matched)} exact matches" 
    )

AMZN: 525/527 exact matches
NVDA: 493/493 exact matches
AAPL: 578/578 exact matches
META: 437/438 exact matches
DELL: 59/59 exact matches
BRK-B: 345/347 exact matches
SMCI: 26/26 exact matches
GOOGL: 676/678 exact matches
MSFT: 681/684 exact matches
JNJ: 259/259 exact matches
UNH: 332/333 exact matches
